# Joining Land Registry Price Data with Energy Performance Certificates (2015 - 2025)

#### Load Price data

In [1]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)


folder = Path('../data/raw')
file_gen = folder.glob('ppd_data*.csv')
pp_raw = pd.concat([pd.read_csv(f) for f in file_gen])
pp_raw.info()

<class 'pandas.DataFrame'>
Index: 135557 entries, 0 to 23586
Data columns (total 16 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   unique_id             135557 non-null  str  
 1   price_paid            135557 non-null  int64
 2   deed_date             135557 non-null  str  
 3   postcode              135115 non-null  str  
 4   property_type         135557 non-null  str  
 5   new_build             135557 non-null  str  
 6   estate_type           135557 non-null  str  
 7   saon                  11820 non-null   str  
 8   paon                  135557 non-null  str  
 9   street                132302 non-null  str  
 10  locality              57572 non-null   str  
 11  town                  135557 non-null  str  
 12  district              135557 non-null  str  
 13  county                135557 non-null  str  
 14  transaction_category  135557 non-null  str  
 15  linked_data_uri       135557 non-null  str  
dtypes

#### Load EPC data

In [2]:
epc_raw = pd.read_csv('../data/raw/EDC_OXFORDSHIRE_2015_2025.csv', usecols= [
    'certificate_number', 'address1', 'address2', 'address3', 'postcode',
    'posttown', 'address', 'local_authority', 'local_authority_label',
    'built_form', 'construction_age_band', 'current_energy_efficiency',
    'current_energy_rating', 'energy_consumption_current', 'extension_count', 'floor_height', 'heating_cost_current',
    'inspection_date', 'lodgement_date', 'number_habitable_rooms',
    'number_heated_rooms', 'potential_energy_efficiency', 'property_type',
    'tenure', 'total_floor_area', 'transaction_type', 'uprn', 'uprn_source'
])
epc_raw['inspection_date'] = epc_raw['inspection_date'].astype('datetime64[ns]')
epc_raw['lodgement_date'] = epc_raw['lodgement_date'].astype('datetime64[ns]')
epc_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 216054 entries, 0 to 216053
Data columns (total 28 columns):
 #   Column                       Non-Null Count   Dtype         
---  ------                       --------------   -----         
 0   certificate_number           216054 non-null  str           
 1   address1                     216054 non-null  str           
 2   address2                     117532 non-null  str           
 3   address3                     27304 non-null   str           
 4   postcode                     216054 non-null  str           
 5   posttown                     213327 non-null  str           
 6   address                      216054 non-null  str           
 7   local_authority              216054 non-null  str           
 8   local_authority_label        216054 non-null  str           
 9   built_form                   213548 non-null  str           
 10  construction_age_band        216054 non-null  str           
 11  current_energy_efficiency    216054 n

In [3]:
print(pp_raw[['postcode','saon', 'paon', 'street', 'locality', 'town']].head())
print(epc_raw[['postcode', 'address3', 'address2', 'address1', 'posttown']].head())

   postcode           saon                  paon street   locality       town
0  HP18 9UH   PARK COTTAGE   CHILLING PLACE FARM    NaN      BRILL  AYLESBURY
1  HP18 9UY            NaN         OAKCROFT FARM    NaN  BOARSTALL  AYLESBURY
2  HP18 9UZ            NaN         TUTHER CORNER    NaN      BRILL  AYLESBURY
3  HP18 9UZ            NaN         TUTHER CORNER    NaN      BRILL  AYLESBURY
4  HP18 9XA  HEADLEY HOUSE  PIDDINGTON GATE FARM    NaN      BRILL  AYLESBURY
   postcode address3                address2                  address1         posttown
0   OX4 3UJ      NaN                     NaN            10 Newman Road           OXFORD
1  OX33 1NL      NaN                Wheatley           10 Barlow Close           OXFORD
2   OX4 3PG      NaN                     NaN          65 Campbell Road           OXFORD
3   OX7 6BE      NaN  Shipton-Under-Wychwood     12 St. Michaels Close  CHIPPING NORTON
4  OX16 5BG      NaN            Tramway Road  Flat 10, Canalside House          BANBURY


In [4]:
counts = pp_raw['postcode'].value_counts()
print(counts[counts == 10])

postcode
MK18 4AS    10
OX15 0PT    10
OX15 0QT    10
OX15 0RF    10
OX15 0RU    10
            ..
OX7 6DY     10
OX7 6DZ     10
OX7 6SY     10
OX7 7BX     10
OX7 7HD     10
Name: count, Length: 648, dtype: int64


In [5]:
print(pp_raw[pp_raw['postcode'] == 'HP18 9UH'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].head())
print(epc_raw[epc_raw['postcode'] == 'HP18 9UH'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].head())

   postcode locality street          saon                 paon       town
0  HP18 9UH    BRILL    NaN  PARK COTTAGE  CHILLING PLACE FARM  AYLESBURY
       postcode address3             address2                       address1   posttown                                   address
27420  HP18 9UH      NaN                Brill  Oak Lodge Chilling Place Farm  AYLESBURY      Oak Lodge Chilling Place Farm, Brill
60325  HP18 9UH    Brill  Chilling Place Farm                   Park Cottage  AYLESBURY  Park Cottage, Chilling Place Farm, Brill


In [6]:
print(pp_raw[pp_raw['postcode'] == 'OX12 9RG'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].sort_values('saon', ascending=True).head(10))
print(epc_raw[epc_raw['postcode'] == 'OX12 9RG'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].sort_values('address1', ascending=True).head(10))

       postcode        locality        street saon        paon     town
10869  OX12 9RG  LETCOMBE REGIS  SOUTH STREET    1  BROOK VIEW  WANTAGE
10870  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   10  BROOK VIEW  WANTAGE
10871  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   11  BROOK VIEW  WANTAGE
10872  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   11  BROOK VIEW  WANTAGE
10873  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   13  BROOK VIEW  WANTAGE
10874  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   13  BROOK VIEW  WANTAGE
10875  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   15  BROOK VIEW  WANTAGE
10876  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   15  BROOK VIEW  WANTAGE
10877  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   16  BROOK VIEW  WANTAGE
10878  OX12 9RG  LETCOMBE REGIS  SOUTH STREET   17  BROOK VIEW  WANTAGE
        postcode        address3      address2       address1 posttown                                      address
205645  OX12 9RG  Letcombe Regis  South Street   1 Brook View  WANTAGE   1 Brook View, South

- Comparison of address data
    - PP full caps vs EPC capital case
    - pp_'locality' is typically epc_'address3' but can be epc_'address2', depending on how the address is split
    - pp_'saon' or pp_'saon'+pp_'paon' can be epc_'address1'
    - pp_'town' == epc_'post_town'
    - Both appear use 'FLAT 8' vs 'Flat 8'

In [7]:
print(pp_raw[pp_raw['postcode'] == 'MK18 4AS'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].sort_values('saon', ascending=False).head(10))
print(epc_raw[epc_raw['postcode'] == 'MK18 4AS'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].sort_values('address1', ascending=False).head(10))

    postcode locality        street saon             paon        town
41  MK18 4AS  FINMERE  FULWELL ROAD  NaN      FIVE GABLES  BUCKINGHAM
42  MK18 4AS  FINMERE  FULWELL ROAD  NaN       HEDGELANDS  BUCKINGHAM
43  MK18 4AS  FINMERE  FULWELL ROAD  NaN        HIGH ACRE  BUCKINGHAM
44  MK18 4AS  FINMERE  FULWELL ROAD  NaN        OAK HOUSE  BUCKINGHAM
45  MK18 4AS  FINMERE  FULWELL ROAD  NaN        OAK HOUSE  BUCKINGHAM
46  MK18 4AS  FINMERE  FULWELL ROAD  NaN  STONEWELL HOUSE  BUCKINGHAM
47  MK18 4AS  FINMERE  FULWELL ROAD  NaN   SYCAMORE HOUSE  BUCKINGHAM
48  MK18 4AS  FINMERE  FULWELL ROAD  NaN   SYCAMORE HOUSE  BUCKINGHAM
49  MK18 4AS  FINMERE  FULWELL ROAD  NaN   SYCAMORE HOUSE  BUCKINGHAM
50  MK18 4AS  FINMERE  FULWELL ROAD  NaN        THE HAVEN  BUCKINGHAM
        postcode address3      address2         address1    posttown                                address
175699  MK18 4AS  Finmere  Fulwell Road      Top Gardens  BUCKINGHAM     Top Gardens, Fulwell Road, Finmere
53071   MK18 4

- Matches well on postcode and paon = address1, saon is null
- One record where posttown uses the value put in address3 for the other records

In [8]:
print(pp_raw[pp_raw['postcode'] == 'OX15 0PT'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].sort_values('paon', ascending=False).head(10))
print(epc_raw[epc_raw['postcode'] == 'OX15 0PT'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].sort_values('address1', ascending=False).head(10))

     postcode locality        street saon                paon     town
300  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN         THE WILLOWS  BANBURY
298  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN       ORCHARD HOUSE  BANBURY
299  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN       ORCHARD HOUSE  BANBURY
297  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN  OLD WESLYAN CHAPEL  BANBURY
296  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN           HOMELANDS  BANBURY
295  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN        HADLEY HOUSE  BANBURY
294  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN       BAMFORD HOUSE  BANBURY
293  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN       APPLE COTTAGE  BANBURY
302  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN                   5  BANBURY
301  OX15 0PT  CLIFTON  CHAPEL CLOSE  NaN                   4  BANBURY
        postcode address3      address2            address1 posttown                                    address
130406  OX15 0PT  Clifton  Chapel Close       The Homestead  BANBURY       The Homestead, Chapel Close, Cli

Fix trailing whitespace

In [ ]:
# Matches whitespace at the start OR end of the string
# pp_raw['paon'].str.contains(r'^\s|\s$', regex=True, na=False).sum()


Index(['unique_id', 'price_paid', 'deed_date', 'postcode', 'property_type', 'new_build', 'estate_type', 'saon', 'paon', 'street', 'locality', 'town', 'district', 'county', 'transaction_category', 'linked_data_uri'], dtype='str')

In [12]:
pp = pp_raw.copy()

for c in pp.select_dtypes(include=['string']).columns:
    pp[c] = pp[c].str.strip()
    print(f"{c} stripped")

unique_id stripped
deed_date stripped
postcode stripped
property_type stripped
new_build stripped
estate_type stripped
saon stripped
paon stripped
street stripped
locality stripped
town stripped
district stripped
county stripped
transaction_category stripped
linked_data_uri stripped


In [16]:
epc = epc_raw.copy()

for c in epc.select_dtypes(include=['string']).columns:
    epc[c] = epc[c].str.strip()
    print(f"{c} stripped")

certificate_number stripped
address1 stripped
address2 stripped
address3 stripped
postcode stripped
posttown stripped
address stripped
local_authority stripped
local_authority_label stripped
built_form stripped
construction_age_band stripped
current_energy_rating stripped
property_type stripped
tenure stripped
transaction_type stripped
uprn_source stripped


In [15]:
pp['address'] = pp['paon'].str.cat([pp['saon'], pp['street'], pp['locality']], sep=' ', na_rep='')
print(pp['address'])

0          CHILLING PLACE FARM PARK COTTAGE  BRILL
1                        OAKCROFT FARM   BOARSTALL
2                            TUTHER CORNER   BRILL
3                            TUTHER CORNER   BRILL
4        PIDDINGTON GATE FARM HEADLEY HOUSE  BRILL
                           ...                    
23582            THE BUTCHERS GROUND   NORTH LEIGH
23583           TURLEY FARM LOTS 2 3 AND 4  HAILEY
23584                 TURLEY LANE PLOT 154  HAILEY
23585                PLOTS 79-84   LONG HANBOROUGH
23586                         114 GARAGE NEW ROAD 
Name: address, Length: 135557, dtype: str
